In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
import polars as pl
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

# Go up two levels: notebooks/ -> profile_analyzer/ -> hmm/
# The hmm/ directory contains the profile_analyzer module
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from profile_analyzer import HMMClassificationAnalyzer, HMMVisualizationPlotter

In [2]:
RESULTS_FILE = '/projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/results/test_pretty_aas_new_scales_awesome_updated_plots/results_20way_segment_20251215_124004.json'
H_CIRCLE_TEST_FILE = '/projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/results/Updated_H_Circle_Test/results_20way_segment_20251216_103957.json'
N_CIRCLE_TEST_FILE = '/projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/results/Updated_N_Circle_Test/results_20way_segment_20251216_104017.json'

CIRCLE_TEST_FILE = H_CIRCLE_TEST_FILE
SEGMENT_PICKLE = '/projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/data/denoise_segment_dynp_parallel_34_BkP_full_pastor_norm_features_clean_1751_4570_min_size_30_length_iqr_25_75_pretty_25_per_aa.pkl'
PROFILE_CSV = '/projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/results/test_pretty_aas_new_scales_awesome_updated_plots/hmm_profiles_20251215_124004.csv'

TARGET_AAS = ['H']
SEGMENT_RANGE = (5, 30)

In [3]:
analyzer = HMMClassificationAnalyzer(
    results_file=RESULTS_FILE,
    target_aas=TARGET_AAS,
    verbose=True
)


Initialized analyzer for amino acids: ['H']


In [4]:
profiles = analyzer.load_profiles(PROFILE_CSV, load_all_aas=True)


Loading profiles from /projects/Genometechlab/andrew/protein/nano-protein-signal/hmm/vrhmm/results/test_pretty_aas_new_scales_awesome_updated_plots/hmm_profiles_20251215_124004.csv
Loaded 700 profile rows
Columns: ['amino_acid', 'state', 'mean', 'std', 'var']
Amino acids in profiles: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']


In [5]:
amino_acid_df = profiles.profile_df.filter(
    pl.col('amino_acid').is_in(TARGET_AAS)
)

In [6]:
amino_acid_df

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""H""",0,-2.137388,0.266993,0.071285
"""H""",1,-1.820577,0.206205,0.04252
"""H""",2,-1.368066,0.275922,0.076133
"""H""",3,-0.92264,0.304424,0.092674
"""H""",4,-0.464695,0.222938,0.049701
…,…,…,…,…
"""H""",30,-0.810803,0.349112,0.121879
"""H""",31,-1.108021,0.217003,0.04709
"""H""",32,-1.33794,0.222812,0.049645


In [7]:
import polars as pl

# Step 1: Merge states 11 and 12
merge_rows = amino_acid_df.filter(pl.col("state").is_in([11, 12]))
merged_row = merge_rows.select(
    pl.lit("H").alias("amino_acid"),
    pl.lit(11).cast(pl.Int64).alias("state"),
    pl.col("mean").mean(),
    pl.col("std").mean(),
    pl.col("var").mean()
)

result = pl.concat([
    amino_acid_df.filter(pl.col("state") < 11),
    merged_row,
    amino_acid_df.filter(pl.col("state") > 12).with_columns(pl.col("state") - 1)
]).sort("state")

# result now has 34 rows, states 0-33

# Step 2: Insert new row at position 20 (shifting 20-33 to 21-34)
new_row = pl.DataFrame({
    "amino_acid": ["H"],
    "state": [20],
    "mean": [0.553891483],
    "std": [0.3025818],
    "var": [0.3025818 ** 2]
})

final = pl.concat([
    result.filter(pl.col("state") < 20),   # states 0-19 (20 rows)
    new_row,                                 # state 20 (1 row)
    result.filter(pl.col("state") >= 20).with_columns(pl.col("state") + 1)  # states 20-33 become 21-34 (14 rows)
]).sort("state")

# Verify
print(f"Final row count: {len(final)}")  # Should be 35
print(f"States: {final['state'].to_list()}")  # Should be 0-34

Final row count: 35
States: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]


In [8]:
final

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""H""",0,-2.137388,0.266993,0.071285
"""H""",1,-1.820577,0.206205,0.04252
"""H""",2,-1.368066,0.275922,0.076133
"""H""",3,-0.92264,0.304424,0.092674
"""H""",4,-0.464695,0.222938,0.049701
…,…,…,…,…
"""H""",30,-0.810803,0.349112,0.121879
"""H""",31,-1.108021,0.217003,0.04709
"""H""",32,-1.33794,0.222812,0.049645


In [9]:
# Get the rows to merge
merge_rows = final.filter(pl.col("state").is_in([9, 10, 11]))

# Create merged row by averaging, with correct dtypes
merged_row = merge_rows.select(
    pl.lit("H").alias("amino_acid"),
    pl.lit(9).cast(pl.Int64).alias("state"),
    pl.col("mean").mean(),
    pl.col("std").mean(),
    pl.col("var").mean()
)

# Rebuild the dataframe
result = pl.concat([
    final.filter(pl.col("state") < 9),
    merged_row,
    final.filter(pl.col("state") > 11).with_columns(pl.col("state") - 2)
]).sort("state")

In [10]:
result

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""H""",0,-2.137388,0.266993,0.071285
"""H""",1,-1.820577,0.206205,0.04252
"""H""",2,-1.368066,0.275922,0.076133
"""H""",3,-0.92264,0.304424,0.092674
"""H""",4,-0.464695,0.222938,0.049701
…,…,…,…,…
"""H""",28,-0.810803,0.349112,0.121879
"""H""",29,-1.108021,0.217003,0.04709
"""H""",30,-1.33794,0.222812,0.049645


In [11]:
new_row = pl.DataFrame({
    "amino_acid": ["H"],
    "state": [15],
    "mean": [0.8513489723],
    "std": [0.25025818],
    "var": [0.25025818 ** 2]
})

# Insert and increment states >= 20
updated_df = pl.concat([
    result.filter(pl.col("state") < 15),
    new_row,
    result.filter(pl.col("state") >= 15).with_columns(pl.col("state") + 1)
]).sort("state")

In [12]:
updated_df

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""H""",0,-2.137388,0.266993,0.071285
"""H""",1,-1.820577,0.206205,0.04252
"""H""",2,-1.368066,0.275922,0.076133
"""H""",3,-0.92264,0.304424,0.092674
"""H""",4,-0.464695,0.222938,0.049701
…,…,…,…,…
"""H""",29,-0.810803,0.349112,0.121879
"""H""",30,-1.108021,0.217003,0.04709
"""H""",31,-1.33794,0.222812,0.049645


In [13]:
new_row = pl.DataFrame({
    "amino_acid": ["H"],
    "state": [17],
    "mean": [0.31082094875],
    "std": [0.27025818],
    "var": [0.27025818 ** 2]
})

# Insert and increment states >= 20
updated_df = pl.concat([
    updated_df.filter(pl.col("state") < 17),
    new_row,
    updated_df.filter(pl.col("state") >= 17).with_columns(pl.col("state") + 1)
]).sort("state")

In [14]:
updated_df

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""H""",0,-2.137388,0.266993,0.071285
"""H""",1,-1.820577,0.206205,0.04252
"""H""",2,-1.368066,0.275922,0.076133
"""H""",3,-0.92264,0.304424,0.092674
"""H""",4,-0.464695,0.222938,0.049701
…,…,…,…,…
"""H""",30,-0.810803,0.349112,0.121879
"""H""",31,-1.108021,0.217003,0.04709
"""H""",32,-1.33794,0.222812,0.049645


In [15]:
# Assuming `original_df` is your full dataframe and `result` is the updated H data

updated_big_df = pl.concat([
    profiles.profile_df.filter(pl.col("amino_acid") != "H"),
    updated_df
]).sort(["amino_acid", "state"])

In [16]:
updated_big_df.write_csv("../../vrhmm/data/altered_H_PASTOR_test_df.csv")

In [17]:
updated_big_df

amino_acid,state,mean,std,var
str,i64,f64,f64,f64
"""A""",0,-2.123094,0.231011,0.053366
"""A""",1,-1.673416,0.161701,0.026147
"""A""",2,-1.316429,0.180664,0.03264
"""A""",3,-1.160042,0.1622,0.026309
"""A""",4,-0.799529,0.216373,0.046817
…,…,…,…,…
"""Y""",30,-0.608784,0.324918,0.105571
"""Y""",31,-0.986182,0.249883,0.062442
"""Y""",32,-1.38509,0.220075,0.048433
